In [ ]:
import pandas as pd
import requests

API_KEY = "7cb1b1-daf77f-94ed27"

# Two different endpoints:
DOCUMENTS_URL = "https://app.overton.io/documents.php"  # Policy documents
PUBLICATIONS_URL = "https://app.overton.io/articles.php"  # Scholarly articles

df = pd.read_excel('./data/export-Person-2025-11-20.xls')
# Clean the Name column - remove education credentials after comma
df['Name'] = df['Name'].str.split(',').str[0].str.strip()

# Search PUBLICATIONS by affiliation (Pennsylvania State University)
params = {"format": "json","page":1, "open_institution_authors": "Pennsylvania State University __OVSEP__ Le Bao __OVSEP__ le bao", "api_key": API_KEY}
response = requests.get(PUBLICATIONS_URL, params=params)
psu_publications_data = response.json()

#### This one is useful
# Search PUBLICATIONS by affiliation (Pennsylvania State University)
params = {"format": "json","page":1, "open_linked_institution_authors": "Florida State University __OVSEP__ Ezra G Goldstein __OVSEP__ ezra g goldstein", "api_key": API_KEY}
response = requests.get(DOCUMENTS_URL, params=params)
psu_documents_data = response.json()

In [7]:
import time

# Initialize new columns
df['publications_count'] = 0
df['documents_count'] = 0

# Institution to search
institution = "Pennsylvania State University"

# Iterate through each researcher
for index, row in df.iterrows():
    name = row['Name']
    
    # Create the query format: Institution __OVSEP__ Name __OVSEP__ lowercase name
    query_string = f"{institution} __OVSEP__ {name} __OVSEP__ {name.lower()}"
    
    print(f"Processing: {name}")
    
    # Search PUBLICATIONS endpoint
    try:
        params = {
            "format": "json",
            "page": 1,
            "open_institution_authors": query_string,
            "api_key": API_KEY
        }
        response = requests.get(PUBLICATIONS_URL, params=params)
        pub_data = response.json()
        pub_count = pub_data.get('query', {}).get('total_results', 0)
        df.at[index, 'publications_count'] = pub_count
        print(f"  Publications: {pub_count}")
    except Exception as e:
        print(f"  Error getting publications: {e}")
        df.at[index, 'publications_count'] = None
    
    # Search DOCUMENTS endpoint
    try:
        params = {
            "format": "json",
            "page": 1,
            "open_linked_institution_authors": query_string,
            "api_key": API_KEY
        }
        response = requests.get(DOCUMENTS_URL, params=params)
        doc_data = response.json()
        doc_count = doc_data.get('query', {}).get('total_results', 0)
        df.at[index, 'documents_count'] = doc_count
        print(f"  Documents: {doc_count}")
    except Exception as e:
        print(f"  Error getting documents: {e}")
        df.at[index, 'documents_count'] = None
    
    # Small delay to be respectful to the API
    time.sleep(0.5)
    print()

print("Processing complete!")

Processing: Uygar Abaci
  Publications: 0
  Publications: 0
  Documents: 0
  Documents: 0

Processing: Ashraf Abdelhalim

Processing: Ashraf Abdelhalim
  Publications: 0
  Publications: 0
  Documents: 0
  Documents: 0

Processing: Khaled Khalil Abdou

Processing: Khaled Khalil Abdou
  Publications: 0
  Publications: 0
  Documents: 0
  Documents: 0

Processing: Muhammad Abdulbasit

Processing: Muhammad Abdulbasit


KeyboardInterrupt: 

In [ ]:
# View the results
df[['Name', 'publications_count', 'documents_count']]

In [41]:
for doc in psu_documents_data['results']:
    print(doc['source']['country'])

Germany
IGO
USA
IGO
Norway
USA
USA
Germany
Germany
UK
Netherlands
Germany
Germany
USA
Germany
IGO
USA
USA
IGO
IGO
USA
USA
IGO
IGO
IGO
IGO
IGO
Netherlands
Netherlands
Netherlands
Netherlands
EU
USA
USA
IGO
Germany
Germany
Germany
IGO
USA
Germany
UK
Germany
Canada
Canada
Canada
Canada
Canada
Chile
IGO


In [35]:
import json

with open('psu_publications_data.json', 'w') as f:
    json.dump(psu_publications_data, f, indent=4)

with open('psu_documents_data.json', 'w') as f:
    json.dump(psu_documents_data, f, indent=4)